# brain_forecast — First Run on Nibi

Minimal end-to-end test + resource profiling so you can size the SLURM allocation.

**Order of this notebook matters.** GPU lock (cell 1) must run before anything imports TensorFlow-bearing libraries, per the Nibi developer guide. `brain_forecast` itself does not import TF, but we keep the guide's pattern in case the movie-decomposition package is imported in the same kernel later.

**Data model (see `AGREEMENTS.md`).** Three separate tables:
- **brain** (spine, required) — keys `(cohort, sub, movie, start)`, carries brain history + targets
- **stimulus** (optional) — keys `(movie, start)`, carries the film features
- **static** (optional) — keys `(cohort, sub)`, one row per subject, carries demographics

Key facts the loader relies on: `sub` is **only unique within a cohort** (so the true identity is `(cohort, sub)`, built internally as `__uid__`); a **cohort can contain several movies**; a **subject can watch several movies**; `start` **restarts at 0 per movie**. Joins are key-based and validated — a misaligned grid raises instead of silently corrupting. CV holds out whole `(cohort, sub)` subjects and stratifies by `movie`.

## 1. Environment + GPU lock
Follows the Nibi dev guide exactly. On a **GPU node** keep `torch.zeros(1).cuda()`. On the **login node** comment it out and set `CUDA_VISIBLE_DEVICES=''`.

In [1]:
import os
os.environ.pop('SSL_CERT_FILE', None)
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CUDA_VISIBLE_DEVICES'] = '-1'      # TF (if ever imported) -> CPU
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'          # PyTorch -> GPU 0

import torch
torch.zeros(1).cuda()  # lock PyTorch onto the GPU FIRST  (comment out on login node)
print('CUDA available:', torch.cuda.is_available())
print('Device        :', torch.cuda.get_device_name(0))
print('VRAM total    : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1e9))

CUDA available: True
Device        : AMD Instinct MI300A
VRAM total    : 128.8 GB


## 2. Install brain_forecast (once per container/overlay)
Skip if already installed. `--no-deps` because the container/def already has every dependency — this avoids pip touching the pinned numpy.

In [2]:
# Adjust the path to wherever you unpacked the package.
import subprocess, sys
BRAIN_FORECAST_DIR = '/home/tamires/projects/rpp-aevans-ab/tamires/BrainForecast'
sys.path.insert(0, BRAIN_FORECAST_DIR)
import brain_forecast; print('brain_forecast', brain_forecast.__version__)

brain_forecast 0.1.0


## 3. Resource probe — helpers
Call `snapshot('label')` at any point. At the end we print peak RAM and peak VRAM, which are what you size `--mem` and the GPU MIG slice against.

In [3]:
import time, gc, resource
_marks = []
def snapshot(label):
    gc.collect()
    ru = resource.getrusage(resource.RUSAGE_SELF)
    ram_gb = ru.ru_maxrss / (1024**2)        # ru_maxrss is KB on Linux -> GB
    if torch.cuda.is_available():
        vram_gb = torch.cuda.max_memory_allocated() / 1e9
        vram_now = torch.cuda.memory_allocated() / 1e9
    else:
        vram_gb = vram_now = 0.0
    _marks.append((label, time.time(), ram_gb, vram_gb, vram_now))
    print(f'[{label:24s}] peakRAM={ram_gb:6.2f}GB  peakVRAM={vram_gb:6.2f}GB  nowVRAM={vram_now:5.2f}GB')

snapshot('start')

[start                   ] peakRAM=  2.29GB  peakVRAM=  0.00GB  nowVRAM= 0.00GB


## 4. Point at your data — three tables

| Table | File | Key columns | Carries |
|---|---|---|---|
| **brain** (spine, required) | `brain.parquet` | `cohort`, `sub`, `movie`, `start` | `observed_dynamic` (brain history) **and** the target columns |
| **stimulus** (optional) | `stimulus.parquet` | `movie`, `start` | `known_dynamic` (movie features) — one set of rows per movie, shared across all subjects |
| **static** (optional) | `static.parquet` | `cohort`, `sub` | `static` (demographics) — exactly one row per `(cohort, sub)` |

For v0 you assemble each of these three tables yourself (concatenating your 10 per-movie stimulus files into one stimulus table, etc.). The loader only does the **cross-channel key joins** and validates them.

Build the typed role lists by column-name pattern — do not hand-list ~6000 columns. Note **where each role lives**: `static` in the static table, `known_dynamic` in the stimulus table, `observed_dynamic` + targets in the brain table.

In [11]:
import pyarrow.parquet as pq
import pandas as pd

BASE = '/home/tamires/projects/rpp-aevans-ab/tamires'

BRAIN_PATH    = f'{BASE}/BrainForecast/inputs/brain_observed_dynamic_per_second.parquet'      # required: cohort, sub, movie, start, umap*, UDFC_*
STIMULUS_PATH = f'{BASE}/BrainForecast/inputs/known_stimuli_per_second.parquet'   # optional: movie, start, mov_*   (None for brain-only)
STATIC_PATH   = f'{BASE}/BrainForecast/inputs/static_participant_features.csv'     # optional: cohort, sub, age, ... (None for brain-only)

brain_cols = pq.ParquetFile(BRAIN_PATH).schema.names
stim_cols  = pq.ParquetFile(STIMULUS_PATH).schema.names if STIMULUS_PATH else []
stat_cols  = pd.read_csv(STATIC_PATH, nrows=0).columns.tolist()  if STATIC_PATH   else []
print(f'brain cols={len(brain_cols)}  stimulus cols={len(stim_cols)}  static cols={len(stat_cols)}')

# ---- EDIT THESE PATTERNS to match your real column names -----------------
# static lives in the STATIC table; known_dynamic in the STIMULUS table;
# observed_dynamic + targets in the BRAIN table.
static            = [c for c in stat_cols  if c in ('age', 'sex', 'handedness')]
static_categorical= [c for c in static     if c in ('sex', 'handedness')]
known_dynamic     = [c for c in stim_cols  if c.startswith('mov_')]           
observed_dynamic  = [c for c in brain_cols if c.startswith('b_')]   
target_cols       = [c for c in brain_cols if c.startswith('b_')]           
# --------------------------------------------------------------------------

print(f'static={len(static)}  known={len(known_dynamic)}  '
      f'observed={len(observed_dynamic)}  targets={len(target_cols)}')

brain cols=115  stimulus cols=4949  static cols=6
static=2  known=4947  observed=111  targets=111


## 5. First experiment — START SMALL
Few targets, few horizons, benchmarks only (no TFT yet). Confirms the three tables join cleanly on their keys, CV splits cleanly at the `(cohort, sub)` level, and gives a RAM baseline before any GPU time.

**`stimulus_join`** controls how the stimulus aligns to the brain on `(movie, start)`:
- `'exact'` (default) — brain and stimulus share the same per-movie `start` grid (same TR, same origin). A brain row with no exact stimulus match **raises**, so a grid mismatch fails loudly instead of producing silent NaNs.
- `'asof'` — nearest `start` per movie, for when the stimulus is sampled on a different grid than the fMRI. Bound it with `asof_tolerance_sec`.

Start with `'exact'`. If it raises a grid-mismatch error, that is information — switch to `'asof'` with a sensible tolerance and record why in `AGREEMENTS.md`.

**Fold count is not fixed at 5.** Any movie with fewer than `loso_threshold` subjects triggers leave-one-subject-out, and the global fold count becomes the max across movies. To force exactly 5 folds everywhere, set `loso_threshold` at or below the smallest movie's subject count.

In [ ]:
from brain_forecast.data import load_bundle_multi
from brain_forecast.features import TabularAdapter, SequenceAdapter
from brain_forecast.cv import StratifiedSubjectOutCV
from brain_forecast.evaluation import run_experiment
from brain_forecast.reporting import aggregate_scores, plot_horizon_curves

TARGET_SUBSET = target_cols[:5]   # start with 5 targets, scale up later

bundle = load_bundle_multi(
    brain_path=BRAIN_PATH,
    stimulus_path=STIMULUS_PATH,          # None for a brain-only run
    static_path=STATIC_PATH,              # None for a brain-only run
    target_cols=TARGET_SUBSET,
    task_type='regression',
    static=static, static_categorical=static_categorical,
    known_dynamic=known_dynamic,
    observed_dynamic=observed_dynamic,
    stimulus_join='exact',                # 'asof' if per-movie grids differ
    asof_tolerance_sec=None,              # e.g. 1.0 when stimulus_join='asof'
    on_missing_subjects='error',          # 'drop' or 'warn' to proceed past gaps
)
snapshot('bundle loaded')
print('subjects (cohort/sub):', bundle.n_subjects())
print('cohorts:', list(bundle.cohorts()))
print('movies :', list(bundle.movies()))

### 5b. (Optional) brain-only sanity check
Confirms the harness runs with the brain table alone — no stimulus, no static. Useful as the very first smoke test and as the pure-autoregressive cell from the design doc. Leave `known_dynamic`/`static` empty here since their tables are absent.

In [ ]:
# Uncomment to run the brain-only check.
# bundle_brain_only = load_bundle_multi(
#     brain_path=BRAIN_PATH,
#     target_cols=TARGET_SUBSET,
#     task_type='regression',
#     observed_dynamic=observed_dynamic,
# )
# print('brain-only subjects:', bundle_brain_only.n_subjects(),
#       '| movies:', list(bundle_brain_only.movies()))

In [ ]:
# Inspect the CV split before running: how many folds, and the held-out
# (cohort/sub) composition per fold. Fold count depends on loso_threshold.
cv = StratifiedSubjectOutCV(k_default=5, loso_threshold=10)
print('planned folds:', cv.n_folds(bundle))
for i, (tr, te) in enumerate(cv.split(bundle)):
    comp = (bundle.df[bundle.df['__uid__'].isin(set(te))]
            .groupby('movie')['__uid__'].nunique().to_dict())
    print(f'fold {i}: {len(tr)} train / {len(te)} test subjects | '
          f'test-by-movie: {comp}')

In [ ]:
results = run_experiment(
    bundle=bundle,
    predictor_specs=[
        {'name': 'persistence'},
        {'name': 'moving_average', 'kwargs': {'k': 5}},
        {'name': 'ar', 'kwargs': {'p': 5}},
    ],
    horizons_min=[0, 5, 15],          # 3 horizons to start
    cv=StratifiedSubjectOutCV(k_default=5, loso_threshold=10),
    tabular_adapter=TabularAdapter(ops=['lag', 'rolling', 'target_history', 'hrf'],
                                   k_lag=3, rolling_window=10, target_history_lags=5),
    output_dir='/home/tamires/scratch/bf_first_run',
)
snapshot('benchmarks done')
aggregate_scores(results, 'r2_mean', by=['predictor', 'horizon_min'])

## 6. TFT smoke test (1 horizon, capped epochs)
Only after the benchmarks pass. GPU-heavy; keep it tiny first to get a VRAM number. Reuses the same three-channel `bundle` — the stimulus must be present and typed as `known_dynamic` for the TFT to have a known-future input (a brain-only TFT is degenerate; `load_bundle_multi` will have warned).

In [ ]:
torch.cuda.reset_peak_memory_stats()
results_tft = run_experiment(
    bundle=bundle,
    predictor_specs=[{'name': 'tft', 'kwargs': {
        'max_epochs': 3,            # tiny: just to measure, not to converge
        'hidden_size': 32,
        'attention_head_size': 2,
        'batch_size': 64,
        'device': 'cuda',
    }}],
    horizons_min=[0],               # ONE horizon for the smoke test
    cv=StratifiedSubjectOutCV(k_default=5, loso_threshold=10),
    sequence_adapter=SequenceAdapter(window_min=5.0),
    output_dir='/home/tamires/scratch/bf_first_run_tft',
)
snapshot('tft smoke done')
results_tft.head()

## 7. Resource summary — use this to size SLURM
`peakRAM` → set `--mem` to ~1.5× this. `peakVRAM` → confirm it fits the MIG slice (the `1g.10gb` slice is **10 GB**). The per-step timings let you extrapolate `--time` for the full target/horizon grid.

In [1]:
import pandas as pd
rows = []
for i, (label, t, ram, vram, vnow) in enumerate(_marks):
    dt = 0.0 if i == 0 else t - _marks[i-1][1]
    rows.append({'stage': label, 'dt_sec': round(dt, 1),
                 'peakRAM_GB': round(ram, 2), 'peakVRAM_GB': round(vram, 2)})
prof = pd.DataFrame(rows)
print(prof.to_string(index=False))
print()
peak_ram  = prof['peakRAM_GB'].max()
peak_vram = prof['peakVRAM_GB'].max()
print(f'PEAK RAM : {peak_ram:.2f} GB   -> suggest --mem={int(max(8, peak_ram*1.5))}G')
print(f'PEAK VRAM: {peak_vram:.2f} GB  -> MIG 1g.10gb gives 10 GB; '
      f"{'OK' if peak_vram < 9 else 'TOO BIG, request a larger slice'}")

NameError: name '_marks' is not defined

## 8. Extrapolate to the full grid
Edit the multipliers to your real plan (e.g. all targets, 7 horizons, 5+ folds, 30 epochs).

In [ ]:
# Time of the 1-horizon / 3-epoch TFT smoke run:
tft_dt = next(r['dt_sec'] for r in rows if r['stage'] == 'tft smoke done')

N_HORIZONS_FULL = 7      # e.g. [0,5,10,15,30,45,60]
EPOCH_SCALE     = 30/3   # full epochs vs the 3 used here
TARGET_SCALE    = len(target_cols)/len(TARGET_SUBSET)  # all targets vs the 5 used

est_sec = tft_dt * N_HORIZONS_FULL * EPOCH_SCALE * TARGET_SCALE
print(f'Smoke TFT (1 horizon, 3 epochs, 5 targets): {tft_dt/60:.1f} min')
print(f'Rough full-grid TFT estimate              : {est_sec/3600:.1f} h')
print('Add ~20% headroom; round --time up to the next hour.')